# 📊 Boletín de Ejercicios 3: SQL con Python

## Trabajando con Bases de Datos Relacionales

**Objetivo:** Dominar la integración de SQL con Python para manipular y consultar bases de datos.

**Instrucciones:**
- En este boletín trabajarás con SQLite, una base de datos ligera incluida con Python
- Aprenderás a crear bases de datos, ejecutar consultas SQL y combinar con Pandas
- Los ejercicios simulan escenarios reales de análisis de datos empresariales
- 🟢 = Básico | 🟡 = Intermedio | 🔴 = Avanzado

---

In [ ]:
# Importar librerías necesarias
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("Librerías importadas correctamente")
print(f"Pandas versión: {pd.__version__}")

## 🟢 Ejercicio 1: Tu Primera Base de Datos

Crea una base de datos SQLite llamada `tienda.db` y una tabla `productos` con los siguientes campos:
- `id` (INTEGER, clave primaria)
- `nombre` (TEXT)
- `precio` (REAL)
- `stock` (INTEGER)

Luego:
1. Inserta 5 productos de ejemplo:

| id | nombre | precio | stock |
|----|--------|--------|-------|
| 1 | Laptop | 899.99 | 25 |
| 2 | Mouse | 29.99 | 100 |
| 3 | Teclado | 59.99 | 75 |
| 4 | Monitor | 349.99 | 30 |
| 5 | Webcam | 79.99 | 50 |

2. Consulta todos los productos
3. Muestra los resultados en un DataFrame de Pandas

In [17]:
import sqlite3
import pandas as pd

lima = '\33[38;5;46m'
azul = '\033[94m'
reset = '\033[0m'

# Crear/Conectar a la base de datos y crear la tabla
with sqlite3.connect('tienda.db') as conexion:
    cursor = conexion.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS productos (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            precio REAL,
            stock INTEGER
        )
    ''')
    
    # Insertar los 5 productos 
    productos_ejemplo = [
        (1, 'Laptop', 899.99, 25),
        (2, 'Mouse', 29.99, 100),
        (3, 'Teclado', 59.99, 75),
        (4, 'Monitor', 349.99, 30),
        (5, 'Webcam', 79.99, 50)
    ]
    
    # Usamos INSERT OR IGNORE para que no falle si volvemos a ejecutar el código
    cursor.executemany('INSERT OR IGNORE INTO productos VALUES (?, ?, ?, ?)', productos_ejemplo)
    conexion.commit()

# Consulta todos los productos y muestra en un DataFrame
with sqlite3.connect('tienda.db') as conexion:
    df_productos = pd.read_sql_query('SELECT * FROM productos', conexion)


print(f"\n{azul}     Inventario de productos{reset}")
print(f"{lima}{df_productos}{reset}")


     Inventario de productos
   id   nombre   precio  stock
0   1   Laptop  989.989     20
1   2    Mouse   32.989    100
2   3  Teclado   65.989     75
3   4  Monitor  384.989     30
4   5   Webcam   87.989     50


## 🟢 Ejercicio 2: Consultas Básicas (SELECT, WHERE)

Usando la base de datos del ejercicio anterior:

1. Consulta todos los productos con precio mayor a 50€
2. Consulta productos con stock menor a 20 unidades
3. Consulta el producto más caro
4. Cuenta cuántos productos hay en total

Muestra cada resultado en un DataFrame.

In [18]:
import sqlite3
import pandas as pd

azul = '\033[94m'
lima = '\33[38;5;46m'
reset = '\033[0m'


# Conectamos a la base de datos creada en el ejercicio anterior
with sqlite3.connect('tienda.db') as conexion:
    
    # Productos con precio mayor a 50€
    df_caros = pd.read_sql_query('SELECT * FROM productos WHERE precio > 50', conexion)
    print(f"\n{azul}Productos > 50€{reset}")
    print(f"{lima}{df_caros}{reset}")
    
    # Productos con stock menor a 20 unidades
    df_poco_stock = pd.read_sql_query('SELECT * FROM productos WHERE stock < 20', conexion)
    print(f"\n{azul}Productos con stock < 20{reset}")
    print(f"{lima}{df_poco_stock}{reset}")
    
    # El producto más caro (usando ORDER BY y LIMIT)
    df_mas_caro = pd.read_sql_query('SELECT * FROM productos ORDER BY precio DESC LIMIT 1', conexion)
    print(f"\n{azul}Producto más caro{reset}")
    print(f"{lima}{df_mas_caro}{reset}")
    
    # Cuenta cuántos productos hay en total (usando la función COUNT)
    df_total = pd.read_sql_query('SELECT COUNT(*) AS total_productos FROM productos', conexion)
    print(f"\n{azul}Total de productos{reset}")
    print(f"{lima}{df_total}{reset}")


Productos > 50€
   id   nombre   precio  stock
0   1   Laptop  989.989     20
1   3  Teclado   65.989     75
2   4  Monitor  384.989     30
3   5   Webcam   87.989     50

Productos con stock < 20
Empty DataFrame
Columns: [id, nombre, precio, stock]
Index: []

Producto más caro
   id  nombre   precio  stock
0   1  Laptop  989.989     20

Total de productos
   total_productos
0                5


## 🟢 Ejercicio 3: Actualizar y Eliminar Datos

Realiza las siguientes operaciones:

1. **UPDATE:** Aumenta el precio de todos los productos en un 10%
2. **UPDATE:** Reduce el stock del producto 'Mouse' en 5 unidades
3. **DELETE:** Elimina productos con stock igual a 0
4. Verifica los cambios consultando toda la tabla

In [ ]:
import sqlite3
import pandas as pd

azul = '\033[94m'
lima = '\33[38;5;46m'
reset = '\033[0m'

with sqlite3.connect('tienda.db') as conexion:
    cursor = conexion.cursor()
    
    # UPDATE: Aumentar el precio de todos los productos en un 10%
    cursor.execute('UPDATE productos SET precio = precio * 1.1')
    
    # UPDATE: Reduce el stock del producto 'Mouse' en 5 unidades
    cursor.execute("UPDATE productos SET stock = stock - 5 WHERE nombre = 'Mouse'")
    
    # DELETE: Elimina productos con stock igual a 0
    cursor.execute('DELETE FROM productos WHERE stock = 0')
    
    # Guardamos los cambios antes de consultar
    conexion.commit()
    
    # Verifica los cambios consultando toda la tabla
    df_verificacion = pd.read_sql_query('SELECT * FROM productos', conexion)
    print(f"\n{azul}Tabla tras las actualizaciones y borrados{reset}")
    print(f"{lima}{df_verificacion}{reset}")


Tabla tras las actualizaciones y borrados
   id   nombre     precio  stock
0   1   Laptop  1088.9879     20
1   2    Mouse    36.2879     95
2   3  Teclado    72.5879     75
3   4  Monitor   423.4879     30
4   5   Webcam    96.7879     50


## 🟡 Ejercicio 4: Crear una Base de Datos Completa

Crea una nueva base de datos `empresa.db` con las siguientes tablas:

**Tabla `departamentos`:**
- id (INTEGER PRIMARY KEY)
- nombre (TEXT)
- presupuesto (REAL)

**Tabla `empleados`:**
- id (INTEGER PRIMARY KEY)
- nombre (TEXT)
- departamento_id (INTEGER, FOREIGN KEY → departamentos.id)
- salario (REAL)
- fecha_ingreso (TEXT)


Luego:
1. Crea ambas tablas
2. Inserta 3 departamentos (IT, Ventas, Marketing)
3. Inserta 6 empleados distribuidos en los departamentos
4. Verifica que los datos se insertaron correctamente

In [25]:
import sqlite3
import pandas as pd

azul = '\033[94m'
lima = '\33[38;5;46m'
reset = '\033[0m'

# Conectar y crear ambas tablas
with sqlite3.connect('empresa.db') as conexion:
    cursor = conexion.cursor()
    
    # Habilitar soporte para claves foráneas en SQLite
    cursor.execute('PRAGMA foreign_keys = ON')
    
    # Crear tabla departamentos
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS departamentos (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            presupuesto REAL
        )
    ''')
    
    # Crear tabla empleados con Foreign Key
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS empleados (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            departamento_id INTEGER,
            salario REAL,
            fecha_ingreso TEXT,
            FOREIGN KEY (departamento_id) REFERENCES departamentos(id)
        )
    ''')
    
    # Insertar 3 departamentos
    depto_datos = [
        (1, 'IT', 50000.0),
        (2, 'Ventas', 30000.0),
        (3, 'Marketing', 25000.0)
    ]
    cursor.executemany('INSERT OR IGNORE INTO departamentos VALUES (?,?,?)', depto_datos)
    
    # Insertar 6 empleados
    empleado_datos = [
        (1, 'Ana García', 1, 42000.0, '2023-01-10'),
        (2, 'Carlos López', 2, 38000.0, '2022-05-15'),
        (3, 'María Ruiz', 1, 45000.0, '2023-03-20'),
        (4, 'Juan Pérez', 3, 35000.0, '2023-06-01'),
        (5, 'Laura Sanz', 2, 36000.0, '2023-02-28'),
        (6, 'Pedro Gómez', 1, 55000.0, '2021-11-10')
    ]
    cursor.executemany('INSERT OR IGNORE INTO empleados VALUES (?,?,?,?,?)', empleado_datos)
    
    conexion.commit()

# Verificar inserción
with sqlite3.connect('empresa.db') as conexion:
    print(f"\n{lima}Departamentos{reset}")
    print(pd.read_sql_query('SELECT * FROM departamentos', conexion))
    print(f"\n{lima}Empleados{reset}")
    print(pd.read_sql_query('SELECT * FROM empleados', conexion))


Departamentos
   id     nombre  presupuesto
0   1         IT      50000.0
1   2     Ventas      30000.0
2   3  Marketing      25000.0

Empleados
   id        nombre  departamento_id  salario fecha_ingreso
0   1    Ana García                1  42000.0    2023-01-10
1   2  Carlos López                2  38000.0    2022-05-15
2   3    María Ruiz                1  45000.0    2023-03-20
3   4    Juan Pérez                3  35000.0    2023-06-01
4   5    Laura Sanz                2  36000.0    2023-02-28
5   6   Pedro Gómez                1  55000.0    2021-11-10


## 🟡 Ejercicio 5: Consultas con JOIN

Usando la base de datos `empresa.db` del ejercicio anterior:

1. Realiza un JOIN entre empleados y departamentos para mostrar:
   - Nombre del empleado
   - Salario
   - Nombre del departamento

2. Calcula el salario promedio por departamento

3. Encuentra el departamento con mayor gasto en salarios (suma de salarios)

4. Lista empleados que ganan más que el promedio de su departamento

In [24]:
import sqlite3
import pandas as pd

azul = '\033[94m'
lima = '\33[38;5;46m'
reset = '\033[0m'

with sqlite3.connect('empresa.db') as conexion:
    
    # JOIN entre empleados y departamentos
    query_join = '''
        SELECT e.nombre AS Empleado, e.salario AS Salario, d.nombre AS Departamento
        FROM empleados e
        INNER JOIN departamentos d ON e.departamento_id = d.id
    '''
    print(f"\n{azul}Detalle Empleados y Departamentos{reset}")
    print(pd.read_sql_query(query_join, conexion))
    
    # Salario promedio por departamento
    query_avg = '''
        SELECT d.nombre, AVG(e.salario) AS salario_promedio
        FROM empleados e
        JOIN departamentos d ON e.departamento_id = d.id
        GROUP BY d.nombre
    '''
    print(f"\n{azul}Salario promedio por depto{reset}")
    print(pd.read_sql_query(query_avg, conexion))
    
    # Departamento con mayor gasto en salarios (SUM y LIMIT)
    query_max_gasto = '''
        SELECT d.nombre, SUM(e.salario) AS gasto_total
        FROM empleados e
        JOIN departamentos d ON e.departamento_id = d.id
        GROUP BY d.nombre
        ORDER BY gasto_total DESC
        LIMIT 1
    '''
    print(f"\n{azul}Departamento con mayor gasto{reset}")
    print(pd.read_sql_query(query_max_gasto, conexion))
    
    # Empleados que ganan más que el promedio de su departamento
    query_sub = '''
        SELECT nombre, salario, departamento_id
        FROM empleados e1
        WHERE salario > (
            SELECT AVG(salario) 
            FROM empleados e2 
            WHERE e1.departamento_id = e2.departamento_id
        )
    '''
    print(f"\n{azul}Empleados por encima del promedio de su depto{reset}")
    print(pd.read_sql_query(query_sub, conexion))


Detalle Empleados y Departamentos
       Empleado  Salario Departamento
0    Ana García  42000.0           IT
1  Carlos López  38000.0       Ventas
2    María Ruiz  45000.0           IT
3    Juan Pérez  35000.0    Marketing
4    Laura Sanz  36000.0       Ventas
5   Pedro Gómez  55000.0           IT

Salario promedio por depto
      nombre  salario_promedio
0         IT      47333.333333
1  Marketing      35000.000000
2     Ventas      37000.000000

Departamento con mayor gasto
  nombre  gasto_total
0     IT     142000.0

Empleados por encima del promedio de su depto
         nombre  salario  departamento_id
0  Carlos López  38000.0                2
1   Pedro Gómez  55000.0                1


## 🟡 Ejercicio 6: Insertar Datos desde Pandas

Crea un DataFrame de Pandas con datos de ventas:

```python
ventas_df = pd.DataFrame({
    'fecha': pd.date_range('2025-01-01', periods=20, freq='D'),
    'producto': np.random.choice(['Laptop', 'Mouse', 'Teclado'], 20),
    'cantidad': np.random.randint(1, 10, 20),
    'precio_unitario': np.random.choice([599.99, 19.99, 45.50], 20)
})
```

Luego:
1. Crea una base de datos `ventas.db` con una tabla `ventas`
2. Inserta el DataFrame completo en la tabla usando `.to_sql()`
3. Consulta las primeras 5 filas desde la base de datos
4. Calcula el ingreso total (cantidad × precio_unitario) usando SQL

In [26]:
import sqlite3
import pandas as pd
import numpy as np 

azul = '\033[94m'
lima = '\33[38;5;46m'
reset = '\033[0m' 

# Configuramos la semilla para que los datos aleatorios sean siempre iguales
np.random.seed(42)

# Creamos el DataFrame de Pandas con los datos de ventas
ventas_df = pd.DataFrame({
    'fecha': pd.date_range('2025-01-01', periods=20, freq='D'),
    'producto': np.random.choice(['Laptop', 'Mouse', 'Teclado'], 20),
    'cantidad': np.random.randint(1, 10, 20),
    'precio_unitario': np.random.choice([599.99, 19.99, 45.50], 20)
})

# Crear la base de datos ventas.db y conectar
with sqlite3.connect('ventas.db') as conexion:
    
    # Insertar el DataFrame completo en la tabla 'ventas'
    ventas_df.to_sql('ventas', conexion, if_exists='replace', index=False)
    
    # Consulta las primeras 5 filas desde la base de datos
    df_inicio = pd.read_sql_query('SELECT * FROM ventas LIMIT 5', conexion)
    print(f"\n{azul}Primeras 5 filas de la tabla 'ventas'{reset}")
    print(f"{lima}{df_inicio}{reset}")
    
    # Calcula el ingreso total (cantidad * precio_unitario) usando SQL
    query_ingreso = 'SELECT SUM(cantidad * precio_unitario) AS ingreso_total FROM ventas'
    df_ingreso = pd.read_sql_query(query_ingreso, conexion)
    
    print(f"\n{azul}Ingreso Total calculado con SQL{reset}")
    print(f"{lima}{df_ingreso}{reset}")



Primeras 5 filas de la tabla 'ventas'
                 fecha producto  cantidad  precio_unitario
0  2025-01-01 00:00:00  Teclado         5            19.99
1  2025-01-02 00:00:00   Laptop         1           599.99
2  2025-01-03 00:00:00  Teclado         6            19.99
3  2025-01-04 00:00:00  Teclado         9           599.99
4  2025-01-05 00:00:00   Laptop         1            19.99

Ingreso Total calculado con SQL
   ingreso_total
0       13600.21


## 🔴 Ejercicio 7: Análisis de E-commerce (Caso Real)

Crea una base de datos `ecommerce.db` con 3 tablas relacionadas:

**Tabla `clientes`:**
- cliente_id (INTEGER PRIMARY KEY)
- nombre (TEXT)
- email (TEXT)
- ciudad (TEXT)

**Tabla `productos`:**
- producto_id (INTEGER PRIMARY KEY)
- nombre (TEXT)
- categoria (TEXT)
- precio (REAL)

**Tabla `pedidos`:**
- pedido_id (INTEGER PRIMARY KEY)
- cliente_id (INTEGER, FOREIGN KEY → clientes.cliente_id)
- producto_id (INTEGER, FOREIGN KEY → productos.producto_id)
- cantidad (INTEGER)
- fecha (TEXT)

Luego realiza:
1. Inserta datos de ejemplo (al menos 5 clientes, 8 productos, 15 pedidos)
2. Consulta el historial de compras de un cliente específico
3. Calcula el total gastado por cada cliente
4. Encuentra los 3 productos más vendidos
5. Calcula las ventas totales por categoría de producto

In [30]:
import sqlite3
import pandas as pd

azul = '\033[94m'
lima = '\33[38;5;46m'
reset = '\033[0m'

# Crear base de datos y tablas relacionadas
with sqlite3.connect('ecommerce.db') as conexion:
    cursor = conexion.cursor()
    cursor.execute('PRAGMA foreign_keys = ON') # Activar integridad referencial
    
    # Crear tabla clientes
    cursor.execute('''CREATE TABLE IF NOT EXISTS clientes (
        cliente_id INTEGER PRIMARY KEY,
        nombre TEXT,
        email TEXT,
        ciudad TEXT
    )''')
    
    # Crear tabla productos
    cursor.execute('''CREATE TABLE IF NOT EXISTS productos (
        producto_id INTEGER PRIMARY KEY,
        nombre TEXT,
        categoria TEXT,
        precio REAL
    )''')
    
    # Crear tabla pedidos
    cursor.execute('''CREATE TABLE IF NOT EXISTS pedidos (
        pedido_id INTEGER PRIMARY KEY,
        cliente_id INTEGER,
        producto_id INTEGER,
        cantidad INTEGER,
        fecha TEXT,
        FOREIGN KEY(cliente_id) REFERENCES clientes(cliente_id),
        FOREIGN KEY(producto_id) REFERENCES productos(producto_id)
    )''')

    # Insertar datos
    clientes = [
        (1, 'Ana López', 'ana@email.com', 'Madrid'),
        (2, 'Luis Pérez', 'luis@email.com', 'Barcelona'),
        (3, 'Eva García', 'eva@email.com', 'Valencia'),
        (4, 'Marc Sanz', 'marc@email.com', 'Madrid'),
        (5, 'Sara Ruiz', 'sara@email.com', 'Sevilla')
    ]
    
    productos = [
        (1, 'Laptop', 'IT', 1200), (2, 'Mouse', 'IT', 25), 
        (3, 'Teclado', 'IT', 45), (4, 'Monitor', 'IT', 300),
        (5, 'Silla', 'Hogar', 150), (6, 'Lámpara', 'Hogar', 40),
        (7, 'Auriculares', 'Audio', 80), (8, 'Altavoz', 'Audio', 120)
    ]
    
    # 15 pedidos aleatorios para tener volumen de análisis
    pedidos = [
        (1,1,1,1,'2025-01-10'), (2,2,5,1,'2025-01-11'), (3,1,2,2,'2025-01-12'),
        (4,3,7,1,'2025-01-12'), (5,4,1,1,'2025-01-13'), (6,5,8,1,'2025-01-14'),
        (7,1,4,1,'2025-01-15'), (8,2,2,3,'2025-01-16'), (9,3,5,2,'2025-01-17'),
        (10,4,3,1,'2025-01-18'), (11,5,7,2,'2025-01-19'), (12,1,2,1,'2025-01-20'),
        (13,2,6,4,'2025-01-21'), (14,3,1,1,'2025-01-22'), (15,4,8,2,'2025-01-23')
    ]

    cursor.executemany('INSERT OR IGNORE INTO clientes VALUES(?,?,?,?)', clientes)
    cursor.executemany('INSERT OR IGNORE INTO productos VALUES(?,?,?,?)', productos)
    cursor.executemany('INSERT OR IGNORE INTO pedidos VALUES(?,?,?,?,?)', pedidos)
    conexion.commit()

# Consultas
with sqlite3.connect('ecommerce.db') as conexion:
    
    # Historial de compras de un cliente específico (ej: cliente_id = 1)
    query_historial = '''
        SELECT p.fecha, pr.nombre, p.cantidad 
        FROM pedidos p 
        JOIN productos pr ON p.producto_id = pr.producto_id 
        WHERE p.cliente_id = 1
    '''
    print(f"\n{lima}Historial de compras (Cliente 1){reset}")
    print(pd.read_sql_query(query_historial, conexion))

    # Total gastado por cada cliente
    query_gasto = '''
        SELECT c.nombre, SUM(p.cantidad * pr.precio) AS total_gastado
        FROM clientes c
        JOIN pedidos p ON c.cliente_id = p.cliente_id
        JOIN productos pr ON p.producto_id = pr.producto_id
        GROUP BY c.cliente_id
        ORDER BY total_gastado DESC
    '''
    print(f"\n{lima}Total gastado por cliente{reset}")
    print(pd.read_sql_query(query_gasto, conexion))

    # 4. Los 3 productos más vendidos (por cantidad)
    query_top3 = '''
        SELECT pr.nombre, SUM(p.cantidad) AS unidades_vendidas
        FROM pedidos p
        JOIN productos pr ON p.producto_id = pr.producto_id
        GROUP BY pr.producto_id
        ORDER BY unidades_vendidas DESC
        LIMIT 3
    '''
    print(f"\n{lima}Top 3 productos más vendidos{reset}")
    print(pd.read_sql_query(query_top3, conexion))

    # 5. Ventas totales por categoría de producto
    query_categoria = '''
        SELECT pr.categoria, SUM(p.cantidad * pr.precio) AS total_ventas
        FROM pedidos p
        JOIN productos pr ON p.producto_id = pr.producto_id
        GROUP BY pr.categoria
    '''
    print(f"\n{lima}Ventas por categoría{reset}")
    print(pd.read_sql_query(query_categoria, conexion))


Historial de compras (Cliente 1)
        fecha   nombre  cantidad
0  2025-01-10   Laptop         1
1  2025-01-12    Mouse         2
2  2025-01-15  Monitor         1
3  2025-01-20    Mouse         1

Total gastado por cliente
       nombre  total_gastado
0  Eva García         1580.0
1   Ana López         1575.0
2   Marc Sanz         1485.0
3  Luis Pérez          385.0
4   Sara Ruiz          280.0

Top 3 productos más vendidos
    nombre  unidades_vendidas
0    Mouse                  6
1  Lámpara                  4
2  Altavoz                  3

Ventas por categoría
  categoria  total_ventas
0     Audio         600.0
1     Hogar         610.0
2        IT        4095.0


## 🔴 Ejercicio 8: Funciones de Agregación Avanzadas

Usando la base de datos `ecommerce.db` del ejercicio anterior:

1. Calcula el ticket promedio (monto promedio por pedido)
2. Encuentra el cliente que más pedidos ha realizado
3. Calcula las ventas por mes (agrupa por mes)
4. Identifica productos que nunca se han vendido
5. Calcula el porcentaje de ventas por ciudad

In [31]:
import sqlite3
import pandas as pd

azul = '\033[94m'
lima = '\33[38;5;46m'
reset = '\033[0m'

with sqlite3.connect('ecommerce.db') as conexion:
    
    # Calcula el ticket promedio (monto promedio por pedido)
    query_ticket = '''
        SELECT AVG(p.cantidad * pr.precio) AS ticket_promedio
        FROM pedidos p
        JOIN productos pr ON p.producto_id = pr.producto_id
    '''
    print(f"\n{lima}1. Ticket Promedio{reset}")
    print(pd.read_sql_query(query_ticket, conexion))

    # Encuentra el cliente que más pedidos ha realizado
    query_cliente_top = '''
        SELECT c.nombre, COUNT(p.pedido_id) AS total_pedidos
        FROM clientes c
        JOIN pedidos p ON c.cliente_id = p.cliente_id
        GROUP BY c.cliente_id
        ORDER BY total_pedidos DESC
        LIMIT 1
    '''
    print(f"\n{lima}2. Cliente con más pedidos{reset}")
    print(pd.read_sql_query(query_cliente_top, conexion))

    # Calcula las ventas por mes (agrupa por mes)
    query_mes = '''
        SELECT strftime('%Y-%m', fecha) AS mes, 
               SUM(p.cantidad * pr.precio) AS ventas_mensuales
        FROM pedidos p
        JOIN productos pr ON p.producto_id = pr.producto_id
        GROUP BY mes
        ORDER BY mes
    '''
    print(f"\n{lima}3. Ventas por mes{reset}")
    print(pd.read_sql_query(query_mes, conexion))

    # Identifica productos que nunca se han vendido
    query_no_vendidos = '''
        SELECT pr.nombre, pr.categoria
        FROM productos pr
        LEFT JOIN pedidos p ON pr.producto_id = p.producto_id
        WHERE p.pedido_id IS NULL
    '''
    print(f"\n{lima}4. Productos nunca vendidos{reset}")
    print(pd.read_sql_query(query_no_vendidos, conexion))

    # Calcula el porcentaje de ventas por ciudad
    query_ciudad_pct = '''
        SELECT c.ciudad, 
               SUM(p.cantidad * pr.precio) AS ventas_ciudad,
               ROUND(SUM(p.cantidad * pr.precio) * 100.0 / (
                   SELECT SUM(p2.cantidad * pr2.precio) 
                   FROM pedidos p2 
                   JOIN productos pr2 ON p2.producto_id = pr2.producto_id
               ), 2) AS porcentaje
        FROM clientes c
        JOIN pedidos p ON c.cliente_id = p.cliente_id
        JOIN productos pr ON p.producto_id = pr.producto_id
        GROUP BY c.ciudad
    '''
    print(f"\n{lima}5. Porcentaje de ventas por ciudad{reset}")
    print(pd.read_sql_query(query_ciudad_pct, conexion))


1. Ticket Promedio
   ticket_promedio
0       353.666667

2. Cliente con más pedidos
      nombre  total_pedidos
0  Ana López              4

3. Ventas por mes
       mes  ventas_mensuales
0  2025-01            5305.0

4. Productos nunca vendidos
Empty DataFrame
Columns: [nombre, categoria]
Index: []

5. Porcentaje de ventas por ciudad
      ciudad  ventas_ciudad  porcentaje
0  Barcelona          385.0        7.26
1     Madrid         3060.0       57.68
2    Sevilla          280.0        5.28
3   Valencia         1580.0       29.78


## 🔴 Ejercicio 9: Integración Pandas + SQL

Combina el poder de Pandas con SQL:

1. Lee la tabla `pedidos` en un DataFrame de Pandas
2. Usando Pandas, calcula estadísticas descriptivas de las cantidades vendidas
3. Crea una nueva columna `ingreso_total` (cantidad × precio)
4. Filtra pedidos con ingreso > 500€ usando Pandas
5. Guarda estos pedidos de alto valor en una nueva tabla `pedidos_premium`
6. Verifica con una consulta SQL que la tabla se creó correctamente

In [33]:
import sqlite3
import pandas as pdç

azul = '\033[94m'
lima = '\33[38;5;46m'
reset = '\033[0m'

# Lee la tabla 'pedidos' en un DataFrame de Pandas
with sqlite3.connect('ecommerce.db') as conexion:
    # Traemos también el precio de la tabla productos para el cálculo posterior
    query = '''
        SELECT p.*, pr.precio 
        FROM pedidos p 
        JOIN productos pr ON p.producto_id = pr.producto_id
    '''
    df_pedidos = pd.read_sql_query(query, conexion)

# Usando Pandas, calcula estadísticas descriptivas de las cantidades vendidas
print(f"\n{lima}Estadísticas descriptivas de cantidades{reset}")
print(df_pedidos['cantidad'].describe())

#Crea una nueva columna 'ingreso_total' (cantidad x precio)
df_pedidos['ingreso_total'] = df_pedidos['cantidad'] * df_pedidos['precio']

# Filtra pedidos con ingreso > 500€ usando Pandas
df_premium = df_pedidos[df_pedidos['ingreso_total'] > 500].copy()

# Guarda estos pedidos de alto valor en una nueva tabla 'pedidos_premium'
with sqlite3.connect('ecommerce.db') as conexion:
    # Usamos to_sql para crear la tabla automáticamente desde el DataFrame
    df_premium.to_sql('pedidos_premium', conexion, if_exists='replace', index=False)
    print(f"\n{azul}Tabla 'pedidos_premium' guardada correctamente.{reset}")

# Verifica con una consulta SQL que la tabla se creó correctamente
with sqlite3.connect('ecommerce.db') as conexion:
    verificacion = pd.read_sql_query('SELECT * FROM pedidos_premium', conexion)
    print(f"\n{azul}Contenido de la nueva tabla 'pedidos_premium'{reset}")
    print(f"{lima}{verificacion}{reset}")


Estadísticas descriptivas de cantidades
count    15.000000
mean      1.600000
std       0.910259
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       4.000000
Name: cantidad, dtype: float64

Tabla 'pedidos_premium' guardada correctamente.

Contenido de la nueva tabla 'pedidos_premium'
   pedido_id  cliente_id  producto_id  cantidad       fecha  precio  \
0          1           1            1         1  2025-01-10  1200.0   
1          5           4            1         1  2025-01-13  1200.0   
2         14           3            1         1  2025-01-22  1200.0   

   ingreso_total  
0         1200.0  
1         1200.0  
2         1200.0  


## 🔴 Ejercicio 10: Dashboard de Métricas Empresariales

Crea un análisis completo tipo dashboard empresarial:

Usando todas las tablas de `ecommerce.db`, genera un informe que incluya:

1. **KPIs Generales:**
   - Total de clientes
   - Total de productos
   - Total de pedidos
   - Ingreso total

2. **Análisis de Clientes:**
   - Top 3 clientes por gasto total
   - Distribución de clientes por ciudad
   - Tasa de retención (clientes con más de 1 pedido)

3. **Análisis de Productos:**
   - Top 5 productos por ingresos
   - Categoría más vendida
   - Precio promedio de venta por categoría

4. **Análisis Temporal:**
   - Ventas por día de la semana
   - Tendencia de ventas (primera vs. última semana)

Presenta todo en un formato claro y profesional.

In [37]:
import sqlite3
import pandas as pd

azul = '\033[94m'
lima = '\33[38;5;46m'
reset = '\033[0m'

print(f"{azul}=== DASHBOARD DE MÉTRICAS EMPRESARIALES (ecommerce.db) ==={reset}")

with sqlite3.connect('ecommerce.db') as conn:
    
    # KPIs GENERALES 
    # Usamos subconsultas simples para obtener totales globales en una sola fila
    query_kpis = '''
        SELECT 
            (SELECT COUNT(*) FROM clientes) as total_clientes,
            (SELECT COUNT(*) FROM productos) as total_productos,
            (SELECT COUNT(*) FROM pedidos) as total_pedidos,
            (SELECT SUM(p.cantidad * pr.precio) FROM pedidos p JOIN productos pr ON p.producto_id = pr.producto_id) as ingreso_total
    '''
    df_kpis = pd.read_sql_query(query_kpis, conn)
    print(f"\n{lima}1. KPIs GENERALES{reset}")
    print(df_kpis.to_string(index=False))

    # ANÁLISIS DE CLIENTES 
    print(f"\n{lima}2. ANÁLISIS DE CLIENTES{reset}")
    
    # Top 3 gasto
    query_top_clientes = '''
        SELECT c.nombre, SUM(p.cantidad * pr.precio) as gasto
        FROM clientes c JOIN pedidos p ON c.cliente_id = p.cliente_id
        JOIN productos pr ON p.producto_id = pr.producto_id
        GROUP BY c.cliente_id ORDER BY gasto DESC LIMIT 3
    '''
    print(f"{azul}- Top 3 Clientes por Gasto{reset}")
    print(pd.read_sql_query(query_top_clientes, conn).to_string(index=False))
    
    # Tasa de retención (clientes con más de 1 pedido)
    query_retencion = '''
        SELECT 
            (SELECT COUNT(*) FROM (SELECT cliente_id FROM pedidos GROUP BY cliente_id HAVING COUNT(*) > 1)) * 100.0 / 
            (SELECT COUNT(DISTINCT cliente_id) FROM pedidos) as tasa_retencion_pct
    '''
    print(f"\n{azul}- Tasa de Retención:{reset} {pd.read_sql_query(query_retencion, conn).iloc[0,0]:.2f}%")

    #3. ANÁLISIS DE PRODUCTOS
    print(f"\n{lima}3. ANÁLISIS DE PRODUCTOS{reset}")
    
    query_prod_metrics = '''
        SELECT pr.categoria, 
               SUM(p.cantidad * pr.precio) as ingresos,
               AVG(pr.precio) as precio_promedio
        FROM productos pr
        LEFT JOIN pedidos p ON pr.producto_id = p.producto_id
        GROUP BY pr.categoria
        ORDER BY ingresos DESC
    '''
    df_prod = pd.read_sql_query(query_prod_metrics, conn)
    print(f"{azul}- Métricas por Categoría:{reset}")
    print(df_prod.to_string(index=False))

    # 4. ANÁLISIS TEMPORAL 

    print(f"\n{lima}4. ANÁLISIS TEMPORAL{reset}")
    query_temporal = '''
        SELECT 
            CASE strftime('%w', fecha)
                WHEN '0' THEN 'Domingo' WHEN '1' THEN 'Lunes' WHEN '2' THEN 'Martes'
                WHEN '3' THEN 'Miércoles' WHEN '4' THEN 'Jueves' WHEN '5' THEN 'Viernes'
                ELSE 'Sábado' END as dia_semana,
            SUM(p.cantidad * pr.precio) as ventas
        FROM pedidos p
        JOIN productos pr ON p.producto_id = pr.producto_id
        GROUP BY dia_semana
        ORDER BY ventas DESC
    '''
    print(f"{azul}- Ventas por Día de la Semana:{reset}")
    print(pd.read_sql_query(query_temporal, conn).to_string(index=False))

=== DASHBOARD DE MÉTRICAS EMPRESARIALES (ecommerce.db) ===

1. KPIs GENERALES
 total_clientes  total_productos  total_pedidos  ingreso_total
              5                8             15         5305.0

2. ANÁLISIS DE CLIENTES
- Top 3 Clientes por Gasto
    nombre  gasto
Eva García 1580.0
 Ana López 1575.0
 Marc Sanz 1485.0

- Tasa de Retención: 100.00%

3. ANÁLISIS DE PRODUCTOS
- Métricas por Categoría:
categoria  ingresos  precio_promedio
       IT    4095.0       502.500000
    Hogar     610.0       113.333333
    Audio     600.0       100.000000

4. ANÁLISIS TEMPORAL
- Ventas por Día de la Semana:
dia_semana  ventas
   Viernes  1500.0
 Miércoles  1500.0
     Lunes  1225.0
    Jueves   315.0
   Domingo   290.0
    Martes   280.0
    Sábado   195.0


## 🔴 Ejercicio BONUS: Transacciones y Manejo de Errores

Implementa una función robusta para transferir stock entre productos:

1. Crea una función `transferir_stock(conn, producto_origen, producto_destino, cantidad)` que:
   - Use transacciones (`BEGIN TRANSACTION`, `COMMIT`, `ROLLBACK`)
   - Verifique que el producto origen tenga suficiente stock
   - Reste la cantidad del origen
   - Sume la cantidad al destino
   - Si algo falla, haga rollback

2. Maneja excepciones apropiadamente
3. Prueba la función con casos exitosos y fallidos

**Pista:** Usa `try-except-finally` y `conn.commit()` / `conn.rollback()`

In [38]:
import sqlite3

azul = '\033[94m'
lima = '\33[38;5;46m'
reset = '\033[0m'

def transferir_stock(conn, id_origen, id_destino, cantidad):
    cursor = conn.cursor()
    
    try:
        # Iniciar transacción
        # En sqlite3, al ejecutar el primer comando se inicia automáticamente, 
        # pero controlamos el final con commit o rollback
        
        #Verificar stock del origen
        cursor.execute('SELECT nombre, stock FROM productos WHERE id = ?', (id_origen,))
        producto = cursor.fetchone()
        
        if producto is None:
            raise Exception(f"Error: El producto con ID {id_origen} no existe.")
        
        nombre_origen, stock_actual = producto
        
        if stock_actual < cantidad:
            # Forzamos un error si no hay stock suficiente
            raise Exception(f"Error: Stock insuficiente en '{nombre_origen}' (Disponible: {stock_actual}).")

        # Restar cantidad del origen
        cursor.execute('UPDATE productos SET stock = stock - ? WHERE id = ?', (cantidad, id_origen))
        
        # Sumar cantidad al destino
        cursor.execute('UPDATE productos SET stock = stock + ? WHERE id = ?', (cantidad, id_destino))

        # Si todo ha ido bien, confirmamos los cambios
        conn.commit()
        print(f"Éxito: Se han transferido {cantidad} unidades del ID {id_origen} al ID {id_destino}.")

    except Exception as e:
        # Si algo falla, deshacemos todo lo hecho en esta transacción
        conn.rollback()
        print(f"Transacción cancelada -> {e}")

# Pruebas

# Usamos la base de datos 'tienda.db' creada en el ejercicio 1
with sqlite3.connect('tienda.db') as conexion:
    
    print(f"\n{lima}--- Prueba 1: Caso Exitoso ---{reset}")
    # Transferir 5 unidades de Laptop (ID 1) a Mouse (ID 2)
    transferir_stock(conexion, 1, 2, 5)
    
    print(f"\n{lima}--- Prueba 2: Fallo por stock insuficiente ---{reset}")
    # Intentar transferir 1000 unidades (ID 1 solo tiene 25 inicialmente)
    transferir_stock(conexion, 1, 2, 1000)
    
    print(f"\n{lima}--- Prueba 3: Fallo por ID inexistente ---{reset}")
    transferir_stock(conexion, 99, 2, 1)

    # Verificación final de saldos
    print(f"\n{lima}Estado final del stock{reset}")
    import pandas as pd
    print(pd.read_sql_query('SELECT id, nombre, stock FROM productos', conexion))


--- Prueba 1: Caso Exitoso ---
Éxito: Se han transferido 5 unidades del ID 1 al ID 2.

--- Prueba 2: Fallo por stock insuficiente ---
Transacción cancelada -> Error: Stock insuficiente en 'Laptop' (Disponible: 15).

--- Prueba 3: Fallo por ID inexistente ---
Transacción cancelada -> Error: El producto con ID 99 no existe.

Estado final del stock
   id   nombre  stock
0   1   Laptop     15
1   2    Mouse    100
2   3  Teclado     75
3   4  Monitor     30
4   5   Webcam     50


---

## 📎 ANEXO: Entender Bloqueos en SQLite

### ¿Qué sucede cuando olvido cerrar una conexión?

A veces, durante el desarrollo o por error, dejamos conexiones abiertas. El impacto depende del tipo de base de datos:

**SQLite (local):**
- El archivo `.db` queda bloqueado y otros procesos no pueden escribir
- Los cambios no se guardan si no ejecutaste `commit()`
- La memoria se libera cuando el programa termina

**Bases de datos remotas (MySQL, Oracle, PostgreSQL):**
- Agota el límite de conexiones simultáneas del servidor
- Consume recursos del servidor (memoria, threads)
- Error típico: "Too many connections"

### Ejercicio práctico: Forzar un bloqueo

Para comprender mejor cómo funcionan los bloqueos, puedes simular uno. Este script muestra cuatro formas:


In [ ]:
# ANEXO: Ejemplos de funciones para forzar bloqueos en SQLite

import sqlite3

# Opción 1: Bloqueo con transacción IMMEDIATE
def bloquear_transaccion(ruta_bd):
    conexion = sqlite3.connect(ruta_bd)
    cursor = conexion.cursor()
    cursor.execute('BEGIN IMMEDIATE')
    cursor.execute('UPDATE productos SET nombre = nombre WHERE id = 1')
    print("BD BLOQUEADA - Presiona Ctrl+C para liberar")
    try:
        while True:
            pass
    except KeyboardInterrupt:
        conexion.rollback()
        conexion.close()

# Opción 2: Bloqueo EXCLUSIVE (total)
def bloquear_exclusivo(ruta_bd):
    conexion = sqlite3.connect(ruta_bd)
    cursor = conexion.cursor()
    cursor.execute('BEGIN EXCLUSIVE')
    print("BD BLOQUEADA TOTALMENTE - Ni lectura ni escritura - Presiona Ctrl+C para liberar")
    try:
        while True:
            pass
    except KeyboardInterrupt:
        conexion.rollback()
        conexion.close()

# Opción 3: Crear archivo de bloqueo
def crear_lock_file(ruta_bd):
    import os
    lock_file = ruta_bd + '-lock'
    with open(lock_file, 'w') as f:
        f.write('BLOQUEADO')
    print(f"Archivo de bloqueo creado: {lock_file} - Presiona Ctrl+C para liberar")
    try:
        while True:
            pass
    except KeyboardInterrupt:
        os.remove(lock_file)
        print(f"Bloqueo liberado")

# Opción 4: Bloqueo con WAL mode
def bloquear_wal(ruta_bd):
    conexion = sqlite3.connect(ruta_bd)
    cursor = conexion.cursor()
    cursor.execute('PRAGMA journal_mode=WAL')
    cursor.execute('BEGIN EXCLUSIVE')
    print("BD bloqueada en WAL mode - Presiona Ctrl+C para liberar")
    try:
        while True:
            pass
    except KeyboardInterrupt:
        conexion.rollback()
        conexion.close()


if __name__ == '__main__':
    ruta_bd = 'empresa.db'
    
    print("Elige una opción:")
    print("1. Bloqueo con transacción IMMEDIATE")
    print("2. Bloqueo EXCLUSIVE (total)")
    print("3. Crear archivo .lock")
    print("4. Bloqueo con WAL mode")
    
    opcion = input("\nOpción (1-4): ").strip()
    
    if opcion == '1':
        bloquear_transaccion(ruta_bd)
    elif opcion == '2':
        bloquear_exclusivo(ruta_bd)
    elif opcion == '3':
        crear_lock_file(ruta_bd)
    elif opcion == '4':
        bloquear_wal(ruta_bd)
    else:
        print("Opción no válida")

Microsoft Windows [Versión 10.0.26200.7462]
(c) Microsoft Corporation. Todos los derechos reservados.

C:\Users\IVANA>cd Desktop

C:\Users\IVANA\Desktop>python bloqueo.py
Elige una opción:
1. Bloqueo con transacción IMMEDIATE
2. Bloqueo EXCLUSIVE (total)
3. Crear archivo .lock
4. Bloqueo con WAL mode

Opción (1-4): 2
BD BLOQUEADA TOTALMENTE - Ni lectura ni escritura - Presiona Ctrl+C para liberar

C:\Users\IVANA\Desktop>




### Comparativa de opciones

| Opción | Tipo | Permite lectura | Permite escritura | Uso |
|--------|------|---|---|---|
| IMMEDIATE | Transacción | ✅ Sí | ❌ No | Simular usuario escribiendo |
| EXCLUSIVE | Transacción | ❌ No | ❌ No | Bloqueo total |
| .lock file | Archivo | ❌ No | ❌ No | Bloqueo manual |
| WAL mode | Modo BD | ✅ Sí | ❌ No | Bloqueo realista |

### Cómo probar en SQLiteStudio

1. Abre una terminal y ejecuta uno de los scripts de bloqueo (opción 2 es la más evidente)
2. Abre SQLiteStudio e intenta hacer una consulta → Verás el error `database is locked`
3. Presiona `Ctrl+C` en la terminal para liberar el bloqueo
4. Prueba de nuevo en SQLiteStudio → Funcionará

### Mejores prácticas para evitar bloqueos

Siempre usa el context manager `with` para garantizar que la conexión se cierre:

```python
# CORRECTO: La conexión se cierra automáticamente
with sqlite3.connect('tienda.db') as conexion:
    cursor = conexion.cursor()
    cursor.execute('SELECT * FROM productos')
    resultados = cursor.fetchall()
# La conexión se cierra aquí automáticamente
```

Esto funciona para **cualquier tipo de base de datos**: SQLite, MySQL, PostgreSQL, Oracle, etc.

---

## ✅ Finalización del Boletín

Se han completado los ejercicios de SQL con Python. Los temas cubiertos incluyen:
- ✅ Creación de bases de datos y tablas
- ✅ Operaciones CRUD (Create, Read, Update, Delete)
- ✅ Consultas SQL (SELECT, WHERE, JOIN, GROUP BY)
- ✅ Funciones de agregación (SUM, AVG, COUNT, MAX, MIN)
- ✅ Integración Pandas-SQL con `.to_sql()` y `.read_sql()`
- ✅ Análisis de datos empresariales
- ✅ Transacciones y manejo de errores

**Siguientes pasos:**
- Practicar con bases de datos más grandes
- Explorar PostgreSQL, MySQL para entornos de producción
- Estudiar SQLAlchemy para ORM avanzado

**Nota importante:** Cerrar siempre las conexiones a bases de datos con `conn.close()`